# EXERCICE 2: Padding

In [3]:
m = b'ceci est une bytestring'

In [9]:
def pad(s):
    nb_octets = 16 - (len(s) % 16)
    return s + b'\x80' + b'\x00' * (nb_octets - 1)

In [8]:
def unpad(s):
    
    i = len(s) - 1
    
    while i >= 0 and s[i] == 0:
        i -= 1
        
    if i < 0 or s[i] != 0x80 or (len(s) - i) > 16:
        raise ValueError("message non correctement paddé")
        
    return s[:i]

In [17]:
# Tests du padding
s1 = b"hello"
p1 = pad(s1)
len(p1)
p1

s2 = b"1234567890123456"  # par exemple 16 octets
p2 = pad(s2)
len(p2)
p2


b'1234567890123456\x80\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'

In [18]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
from Crypto.Random.random import randrange
from Crypto.Util.strxor import strxor as xor
from Crypto.Util.number import bytes_to_long, long_to_bytes
from hashlib import md5, sha3_256, sha1

def Enc(bloc, clef):
    if not isinstance(bloc, bytes) or not isinstance(clef, bytes):
        raise TypeError("le bloc et la clef doivent être de type bytes")
    if len(bloc) != 16 or len(clef) != 16:
        raise ValueError("le bloc et la clef doivent faire 16 octets")
    return AES.new(clef, AES.MODE_ECB).encrypt(bloc)

def Dec(bloc, clef):
    if not isinstance(bloc, bytes) or not isinstance(clef, bytes):
        raise TypeError("le bloc et la clef doivent être de type bytes")
    if len(bloc) != 16 or len(clef) != 16:
        raise ValueError("le bloc et la clef doivent faire 16 octets")
    return AES.new(clef, AES.MODE_ECB).decrypt(bloc)

# EXERCICE 3: Mode ECB

In [32]:
def Enc_ECB(clair, clef):
   
    clair_padde = pad(clair)
    chiffre = b""
    
    for i in range(0, len(clair_padde), 16):
        
        bloc = clair_padde[i:i+16]
        
        chiffre += Enc(bloc, clef)
        
    return chiffre

def Dec_ECB(chiffre, clef):
    
    if len(chiffre) % 16 != 0:
        
        raise ValueError("la taille du chiffré doit être un multiple de 16")
    clair_padde = b""
    
    for i in range(0, len(chiffre), 16):
        
        bloc = chiffre[i:i+16]
        
        clair_padde += Dec(bloc, clef)
        
    return unpad(clair_padde)

In [36]:
clef = get_random_bytes(16)
message = b"Bonjour, ceci est un test de chiffrement en mode ECB !"
chiffre = Enc_ECB(message, clef)
dechiffre = Dec_ECB(chiffre, clef)
print("Message clair :", message)
print("Déchiffré ECB :", dechiffre)
print("Test ECB réussi :", dechiffre == message)

Message clair : b'Bonjour, ceci est un test de chiffrement en mode ECB !'
Déchiffré ECB : b'Bonjour, ceci est un test de chiffrement en mode ECB !'
Test ECB réussi : True


# EXERCICE 4: Mode CBC

In [37]:
def Enc_CBC(clair, clef):
   
    clair_padde = pad(clair)
    
    iv = get_random_bytes(16)
    chiffre = iv
    c_prec = iv
    
    for i in range(0, len(clair_padde), 16):
        bloc = clair_padde[i:i+16]
        
        c_actuel = Enc(xor(bloc, c_prec), clef)
        chiffre += c_actuel
        c_prec = c_actuel
    return chiffre

def Dec_CBC(chiffre, clef):
   
    if len(chiffre) < 16 or len(chiffre) % 16 != 0:
        
        raise ValueError("la taille du chiffré doit être un multiple de 16 au moins égal à 16")
    iv = chiffre[:16]
    c_prec = iv
    clair_padde = b""
    
    for i in range(16, len(chiffre), 16):
        c_actuel = chiffre[i:i+16]
        bloc = xor(Dec(c_actuel, clef), c_prec)
        
        clair_padde += bloc
        
        c_prec = c_actuel
    return unpad(clair_padde)

In [41]:
clef = get_random_bytes(16)



# EXERCICE 5: attaque sur ECB